# DFlash Domain Benchmark (Colab)

Speculative decoding: drafter `z-lab/Qwen3-8B-DFlash-b16` + target `Qwen/Qwen3-8B`.

**GPU requirement:** you need ~20 GB VRAM (bf16 8B target + 1B drafter + KV cache).
- Free Colab **T4 (16 GB) is NOT enough** — the 8B target alone is ~16 GB in bf16.
- Use Colab Pro and pick an **L4 (24 GB)** or **A100 (40 GB)** runtime:
  `Runtime > Change runtime type > GPU > L4/A100`.

This notebook uploads the four benchmark scripts inline, so you don't need to clone anything.

In [ ]:
# 1. Check the GPU
import torch, subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU'
name = torch.cuda.get_device_name(0)
gb = torch.cuda.get_device_properties(0).total_memory/1e9
print(f'GPU: {name}  {gb:.1f} GB')
if gb < 20: print('\n WARNING: <20GB VRAM. Use an L4 or A100 runtime, or expect OOM.')

In [ ]:
# 2. Install pinned deps (Colab usually already has a matching torch CUDA build)
!pip -q install 'transformers==4.57.3' 'accelerate>=1.0.0'
import transformers, torch; print('transformers', transformers.__version__, '| torch', torch.__version__)

## 3. Get the benchmark scripts

Easiest: on the left **Files** panel, upload `prompts.py`, `spec_patch.py`, `benchmark.py`, and `aggregate.py` from the *Benchmarking domains* folder. Then run the cell below to confirm they're present.

(Alternatively `from google.colab import files; files.upload()` and select all four.)

In [ ]:
import os
needed = ['prompts.py', 'spec_patch.py', 'benchmark.py', 'aggregate.py']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print('Missing:', missing, '\nUpload them, or run the upload widget:')
    from google.colab import files; files.upload()
else:
    print('All scripts present:', needed)

In [ ]:
# 4a. QUICK SMOKE TEST — 2 domains x 5 prompts, short outputs (~1-2 min once weights are cached).
#     Verifies the wiring + the lossless correctness check before you commit to a long run.
!python benchmark.py --run-name smoke --limit 5 --max-new-tokens 256 \
    --categories lang_english code_python
!python aggregate.py results/smoke.jsonl

In [ ]:
# 4b. MEDIUM RUN — 20 prompts across every domain (good coverage, ~30-60 min on an L4).
#     Bump --limit to 100 for the full run the task asks for (several hours).
!python benchmark.py --run-name dflash_bench --limit 20 --max-new-tokens 512 --categories all
!python aggregate.py results/dflash_bench.jsonl

In [ ]:
# 5. Show the report and download results
from IPython.display import Markdown, display
display(Markdown(open('results/dflash_bench_report.md').read()))
from google.colab import files
files.download('results/dflash_bench_report.md')
files.download('results/dflash_bench_by_category.csv')
files.download('results/dflash_bench.jsonl')